In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from keras.models import load_model
from keras.layers import concatenate, Dense
from tensorflow.keras import regularizers, Model, Input

from tensorflow.keras.callbacks import Callback, EarlyStopping

## Simulation1a

In [ ]:
X_id = pd.read_excel('Large_Scale_PONG/new_filtered_fasttetx_vectors/data/words_encodings.xlsx', index_col=0, sheet_name="identity").values.astype(np.float32)
X_pos = pd.read_excel('Large_Scale_PONG/new_filtered_fasttetx_vectors/data/words_encodings.xlsx', index_col=0, sheet_name="position").values.astype(np.float32)
X_allog = pd.read_excel('Large_Scale_PONG/new_filtered_fasttetx_vectors/data/words_encodings.xlsx', index_col=0,sheet_name="allog").values.astype(np.float32)
Y = (
    pd.read_excel(
        "Large_Scale_PONG/new_filtered_fasttetx_vectors/data/words_encodings.xlsx",
        sheet_name="onehot",
        index_col=0,
    )
    .to_numpy(dtype=np.float32)
)
in_id   = Input(shape=(35,),  name='in_id')
tower_id  = Dense(700, activation='sigmoid')(in_id)


in_pos  = Input(shape=(100,), name='in_pos')
tower_pos = Dense(700, activation="sigmoid")(in_pos)

in_allog  = Input(shape=(116,), name='in_allog')
tower_allog = Dense(700, activation="sigmoid")(in_allog)

combined = concatenate([tower_id, tower_pos, tower_allog])
h = Dense(500, activation="sigmoid")(combined)
output = Dense(350, activation="sigmoid")(h)

model1 = Model(inputs=[in_id, in_pos, in_allog], outputs=output)


model1.compile(optimizer="adam", loss="mse")
callback = tf.keras.callbacks.EarlyStopping(monitor='loss', patience=10)

model1.fit([X_id, X_pos,X_allog],Y,batch_size=25, epochs=700, callbacks=[callback])
# model1.save('simulation_1a_model.h5')

# Test the model
We will find the correlation distance between the original word and its different modified forms.

In [ ]:
from pathlib import Path
data_lst = [
    'Large_Scale_PONG/new_filtered_fasttetx_vectors/data/TL_words_encodings.xlsx',
    'Large_Scale_PONG/new_filtered_fasttetx_vectors/data/DL_words_encodings.xlsx'
]

# output DataFrame and a dict to hold preds
df_sim1 = pd.DataFrame()
preds = {}


for doc in data_lst:

    key = Path(doc).stem.replace('_words_encodings', '')
    X_id    = pd.read_excel(doc, sheet_name='identity', index_col=0).values.astype(np.float32)
    X_pos   = pd.read_excel(doc, sheet_name='position', index_col=0).values.astype(np.float32)
    X_allog   = pd.read_excel(doc, sheet_name='allog', index_col=0).values.astype(np.float32)
    pred = model1.predict([X_id, X_pos, X_allog])
    preds[key] = pred

    # per-word correlation distances
    valz = []
    for i in range(len(Y)):
        r = np.corrcoef(pred[i], Y[i])[0,1]
        valz.append(round(1-r,3))

    df_sim1[f'target-{key}'] = valz



df_sim1.to_excel('Large_Scale_PONG/new_filtered_fasttetx_vectors/results/PONG_simulation1a_correlation_8.xlsx',index=False)
